# Movies. Classification based on CLS token embeddings

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from rital_nlp_project.common.utils import load_clean_data
from rital_nlp_project.common.notebook_utils import init_notebook
from rital_nlp_project.common.models.models_eval import eval_pipeline_movies
from rital_nlp_project.movies.models_utils import split_fn

init_notebook()

In [2]:
def print_all_metrics(true, decision, threshold, preds=None):
    if preds is None:
        preds = (decision > threshold).astype(int)

    print(f"Accuracy  : {accuracy_score(true, preds):.2f}")
    print(f"Precision : {precision_score(true, preds):.2f}")
    print(f"Recall    : {recall_score(true, preds):.2f}")
    print(f"F1-score  : {f1_score(true, preds):.2f}")
    
    if decision is not None:
        print(f"ROC AUC   : {roc_auc_score(true, decision):.2f}")
    print()

def majority_class(series):
    return series.mode().iloc[0]

In [5]:
# Importing + train/test split
data = pd.read_parquet("../Dataset/embeddings/movies_clean_bert_metadata_bert-base-uncased.parquet")
data_by_example = data.drop_duplicates("example_id")

train_example_ids, test_example_ids, _, _ = split_fn(
    data_by_example["example_id"].values,
    data_by_example["label"].values,
 )

train_mask = data["example_id"].isin(train_example_ids)
X_train = np.stack(data.loc[train_mask, "embedding"].to_numpy())
y_train = data.loc[train_mask, "label"].to_numpy()

test_mask = data["example_id"].isin(test_example_ids)
X_test = np.stack(data.loc[test_mask, "embedding"].to_numpy())
y_test_chunk = data.loc[test_mask, "label"].to_numpy()
test_example_ids_full = data.loc[test_mask, "example_id"].to_numpy()

### Bert with mean agg

#### Eval on train/test split

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

chunk_preds = pd.DataFrame(data={
        "example_id": test_example_ids_full,
        "prob_1": model.predict_proba(X_test)[:, 1],
        "true": y_test_chunk,
    }
)
ex_preds = chunk_preds.groupby("example_id").agg(
    prob_1_mean=("prob_1", "mean"),
    prob_1_max=("prob_1", "max"),
    true=("true", "first"),
)

In [ ]:
print("Mean pooling")
probs = ex_preds["prob_1_mean"].values
print_all_metrics(ex_preds["true"], probs, 0.5)

print("Max pooling")
probs = ex_preds["prob_1_max"].values
print_all_metrics(ex_preds["true"], probs, 0.5)

Mean pooling
Accuracy  : 0.88
Precision : 0.85
Recall    : 0.93
F1-score  : 0.89
ROC AUC   : 0.94

Max pooling
Accuracy  : 0.80
Precision : 0.73
Recall    : 0.94
F1-score  : 0.82
ROC AUC   : 0.92



In [ ]:
pd.DataFrame({"prob_1":probs, "true":})

In [53]:
model = LinearSVC()
model.fit(X_train, y_train)

chunk_preds = pd.DataFrame(
    {
        "example_id": test_example_ids_full,
        "pred":  model.predict(X_test),
        "true": y_test_chunk,
        "decision": model.decision_function(X_test)
    }
)

ex_preds = chunk_preds.groupby("example_id").agg(
    pred=("pred", majority_class),
    true=("true", "first"),
    decision=("decision", "mean")
)

print("Mean pooling")
print_all_metrics(ex_preds["true"], decision=ex_preds["decision"], threshold=0)

print("Majority vote")
print_all_metrics(ex_preds["true"], decision=None, threshold=None, preds=ex_preds["pred"])

Mean pooling
Accuracy  : 0.87
Precision : 0.85
Recall    : 0.90
F1-score  : 0.87
ROC AUC   : 0.94

Majority vote
Accuracy  : 0.87
Precision : 0.88
Recall    : 0.85
F1-score  : 0.87

